In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import ee
import geemap

# Initialize Earth Engine
ee.Initialize(project='peppy-gist-511817-e8')

# Check device (GPU acceleration if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

c:\Users\sagal\anaconda3\envs\deforestation\lib\site-packages\google\api_core\_python_version_support.py:223: FutureWarning: You are using a non-supported Python version (3.10.22). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


Using device: cuda
GPU Model: NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
import torch.nn.functional as F

# 1. Define U-Net Building Blocks (Double Convolution)
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

# 2. Define Full U-Net Model for Binary Deforestation Segmentation
class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=1): # 4 input bands: R, G, B, NIR
        super(UNet, self).__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(256, 128)
        
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(128, 64)
        
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        
        x = self.up1(x3)
        x = torch.cat([x, x2], dim=1)
        x = self.conv1(x)
        
        x = self.up2(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv2(x)
        
        logits = self.outc(x)
        return logits

# Initialize model and move to RTX 5060 GPU
model = UNet(in_channels=4, out_channels=1).to(device)
print("U-Net model initialized and moved to GPU successfully!")
print(model.__class__.__name__)

U-Net model initialized and moved to GPU successfully!
UNet


In [4]:
from torch.utils.data import TensorDataset

# 1. Create synthetic dataset for testing the pipeline (Batch of 8 samples, 4 channels, 256x256 resolution)
# In production, you will replace this with your exported GEE GeoTIFF patches loaded via rasterio or geemap
dummy_images = torch.randn(16, 4, 256, 256) # 16 training samples
dummy_masks = torch.randint(0, 2, (16, 1, 256, 256)).float() # Binary masks (0 or 1)

# Wrap into a PyTorch DataLoader
train_dataset = TensorDataset(dummy_images, dummy_masks)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# 2. Define Loss Function and Optimizer
criterion = nn.BCEWithLogitsLoss() # Standard binary cross-entropy for segmentation logits
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Run a Quick Training Loop to Test GPU Execution
model.train()
epochs = 3

print("Starting quick U-Net training test on RTX 5060...")
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_images, batch_masks in train_loader:
        # Move tensors to GPU
        batch_images = batch_images.to(device)
        batch_masks = batch_masks.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(batch_images)
        
        # Compute loss
        loss = criterion(outputs, batch_masks)
        
        # Backward pass & optimize
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss / len(train_loader):.4f}")

print("U-Net training pipeline executed successfully on your GPU!")

Starting quick U-Net training test on RTX 5060...
Epoch [1/3], Loss: 0.7029
Epoch [2/3], Loss: 0.6891
Epoch [3/3], Loss: 0.6818
U-Net training pipeline executed successfully on your GPU!


BASELINE U_NET

In [11]:
import ee
import geemap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# 1. Initialize and use a focused AOI around an active deforestation cluster
ee.Initialize(project='peppy-gist-511817-e8')

aoi = ee.Geometry.Polygon([[
    [-59.7, -10.4],
    [-59.6, -10.4],
    [-59.6, -10.5],
    [-59.7, -10.5]
]])

# Pull Sentinel-2 imagery for 2024 over the focused AOI
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate('2024-06-01', '2024-10-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

image_2024 = collection.median().clip(aoi)

# Compute baseline change detection mask
image_2022 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate('2022-06-01', '2022-10-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .median().clip(aoi)

ndvi_2022 = image_2022.normalizedDifference(['B8', 'B4'])
ndvi_2024 = image_2024.normalizedDifference(['B8', 'B4'])
ndvi_diff = ndvi_2022.subtract(ndvi_2024)
deforestation_mask = ndvi_diff.gt(0.25)

cleaned_deforestation = (deforestation_mask
    .selfMask()
    .connectedPixelCount(maxSize=50, eightConnected=True)
    .gte(20))

# 2. Extract training tensors
s2_rgbn = image_2024.select(['B4', 'B3', 'B2', 'B8'])
mask_image = cleaned_deforestation.selfMask().unmask(0).rename('label')
training_image = s2_rgbn.addBands(mask_image)

print("Extracting training region from GEE...")
region_array = geemap.ee_to_numpy(training_image, region=aoi, scale=10)

patch_size = 256
h, w, c = region_array.shape
patches_images, patches_masks = [], []

for y in range(0, h - patch_size + 1, patch_size):
    for x in range(0, w - patch_size + 1, patch_size):
        patch = np.nan_to_num(region_array[y:y+patch_size, x:x+patch_size, :])
        patches_images.append(patch[:, :, :4].transpose(2, 0, 1))
        patches_masks.append(patch[:, :, 4:5].transpose(2, 0, 1))

X_train = torch.tensor(np.array(patches_images), dtype=torch.float32)
y_train = torch.tensor(np.array(patches_masks), dtype=torch.float32)
print(f"Training tensors ready! Shape: {X_train.shape}")

# 3. Train Baseline U-Net (5 Epochs) on RTX 5060
real_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=2, shuffle=True)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

print("\nStarting baseline U-Net training (5 epochs)...")
model.train()
for epoch in range(5):
    epoch_loss = 0.0
    for batch_imgs, batch_msks in real_loader:
        batch_imgs, batch_msks = batch_imgs.to(device), batch_msks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch_imgs), batch_msks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch [{epoch+1}/5], Loss: {epoch_loss / len(real_loader):.4f}")

torch.save(model.state_dict(), 'deforestation_unet_model_baseline.pth')
print("\nBaseline training complete! Weights saved.")

Extracting training region from GEE...
Training tensors ready! Shape: torch.Size([16, 4, 256, 256])

Starting baseline U-Net training (5 epochs)...
Epoch [1/5], Loss: 0.1403
Epoch [2/5], Loss: 0.1248
Epoch [3/5], Loss: 0.1116
Epoch [4/5], Loss: 0.1007
Epoch [5/5], Loss: 0.0916

Baseline training complete! Weights saved.


Baseline U-net Metrics

In [12]:
# Evaluate Baseline Model Performance
model.eval()
total_iou, total_dice = 0.0, 0.0
num_samples = X_train.shape[0]
smooth = 1e-6

with torch.no_grad():
    for i in range(num_samples):
        img = X_train[i:i+1].to(device)
        true_mask = y_train[i:i+1].to(device)
        
        pred_logits = model(img)
        pred_probs = torch.sigmoid(pred_logits)
        pred_mask = (pred_probs > 0.5).float()
        
        pred_flat = pred_mask.view(-1)
        true_flat = true_mask.view(-1)
        
        intersection = (pred_flat * true_flat).sum().item()
        union = pred_flat.sum().item() + true_flat.sum().item() - intersection
        
        iou = (intersection + smooth) / (union + smooth)
        dice = (2.0 * intersection + smooth) / (pred_flat.sum().item() + true_flat.sum().item() + smooth)
        
        total_iou += iou
        total_dice += dice

print(f"--- Baseline U-Net Performance Metrics ---")
print(f"Mean Intersection over Union (IoU): {(total_iou / num_samples) * 100:.2f}%")
print(f"Mean Dice Coefficient (F1-Score):   {(total_dice / num_samples) * 100:.2f}%")

--- Baseline U-Net Performance Metrics ---
Mean Intersection over Union (IoU): 44.54%
Mean Dice Coefficient (F1-Score):   45.17%


v2 upgraded U-net model

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import random

# 1. Define Combined Dice + BCE Loss
class DiceBCELoss(nn.Module):
    def __init__(self):
        super(DiceBCELoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets, smooth=1e-6):
        bce_loss = self.bce(inputs, targets)
        sigmoid_inputs = torch.sigmoid(inputs)
        intersection = (sigmoid_inputs * targets).sum(dim=(1, 2, 3))
        total = sigmoid_inputs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice_loss = 1 - (2.0 * intersection + smooth) / (total + smooth)
        return bce_loss + dice_loss.mean()

# 2. Data Augmentation Function (Random Flips)
def augment_batch(images, masks):
    augmented_images, augmented_masks = [], []
    for img, mask in zip(images, masks):
        if random.random() > 0.5:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[2])
        if random.random() > 0.5:
            img = torch.flip(img, dims=[1])
            mask = torch.flip(mask, dims=[1])
        augmented_images.append(img)
        augmented_masks.append(mask)
    return torch.stack(augmented_images), torch.stack(augmented_masks)

# 3. Setup Upgraded Training Configuration
criterion = DiceBCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 15
print("Starting upgraded U-Net training on RTX 5060 (Dice+BCE Loss + Augmentation)...")

model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    aug_imgs, aug_masks = augment_batch(X_train, y_train)
    aug_loader = DataLoader(TensorDataset(aug_imgs, aug_masks), batch_size=4, shuffle=True)
    
    for batch_imgs, batch_msks in aug_loader:
        batch_imgs, batch_msks = batch_imgs.to(device), batch_msks.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_imgs)
        loss = criterion(outputs, batch_msks)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(aug_loader)
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Combined Loss: {avg_loss:.4f}")

# Save the upgraded model weights
torch.save(model.state_dict(), 'deforestation_unet_model_v2.pth')
print("\nUpgraded model training complete! Weights saved to 'deforestation_unet_model_v2.pth'.")

Starting upgraded U-Net training on RTX 5060 (Dice+BCE Loss + Augmentation)...
Epoch [1/15], Combined Loss: 1.0788
Epoch [3/15], Combined Loss: 1.0538
Epoch [6/15], Combined Loss: 1.0263
Epoch [9/15], Combined Loss: 0.9912
Epoch [12/15], Combined Loss: 0.9901
Epoch [15/15], Combined Loss: 0.9536

Upgraded model training complete! Weights saved to 'deforestation_unet_model_v2.pth'.


In [14]:
# 1. Evaluate Upgraded Model Performance
model.eval()
total_iou, total_dice = 0.0, 0.0
num_samples = X_train.shape[0]
smooth = 1e-6

with torch.no_grad():
    for i in range(num_samples):
        img = X_train[i:i+1].to(device)
        true_mask = y_train[i:i+1].to(device)
        
        # Forward pass & prediction using the upgraded weights
        pred_logits = model(img)
        pred_probs = torch.sigmoid(pred_logits)
        pred_mask = (pred_probs > 0.5).float()
        
        # Flatten tensors for metric calculation
        pred_flat = pred_mask.view(-1)
        true_flat = true_mask.view(-1)
        
        # Intersection and Union
        intersection = (pred_flat * true_flat).sum().item()
        union = pred_flat.sum().item() + true_flat.sum().item() - intersection
        
        # IoU and Dice
        iou = (intersection + smooth) / (union + smooth)
        dice = (2.0 * intersection + smooth) / (pred_flat.sum().item() + true_flat.sum().item() + smooth)
        
        total_iou += iou
        total_dice += dice

mean_iou = total_iou / num_samples
mean_dice = total_dice / num_samples

print(f"--- Upgraded U-Net Performance Metrics ---")
print(f"Mean Intersection over Union (IoU): {mean_iou * 100:.2f}%")
print(f"Mean Dice Coefficient (F1-Score):   {mean_dice * 100:.2f}%")

--- Upgraded U-Net Performance Metrics ---
Mean Intersection over Union (IoU): 6.90%
Mean Dice Coefficient (F1-Score):   10.71%


v3 Third trial of U-net lower learning rate = 0.001

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import random

# Trial Identifier
trial_name = "trial_2_dice_bce"
model_save_path = f"deforestation_unet_model_{trial_name}.pth"
print(f"Running Experiment: {trial_name}")

# Re-initialize fresh model weights
model = UNet(in_channels=4, out_channels=1).to(device)

# Stable Combined Loss
class StableDiceBCELoss(nn.Module):
    def __init__(self):
        super(StableDiceBCELoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets, smooth=1e-5):
        bce_loss = self.bce(inputs, targets)
        sigmoid_inputs = torch.sigmoid(inputs)
        intersection = (sigmoid_inputs * targets).sum(dim=(1, 2, 3))
        total = sigmoid_inputs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice_loss = 1 - (2.0 * intersection + smooth) / (total + smooth)
        return bce_loss + dice_loss.mean()

# Safe augmentation function
def augment_batch(images, masks):
    augmented_images, augmented_masks = [], []
    for img, mask in zip(images, masks):
        if random.random() > 0.5:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[2])
        if random.random() > 0.5:
            img = torch.flip(img, dims=[1])
            mask = torch.flip(mask, dims=[1])
        augmented_images.append(img)
        augmented_masks.append(mask)
    return torch.stack(augmented_images), torch.stack(augmented_masks)

criterion = StableDiceBCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

epochs = 12
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    aug_imgs, aug_masks = augment_batch(X_train, y_train)
    aug_loader = DataLoader(TensorDataset(aug_imgs, aug_masks), batch_size=2, shuffle=True)
    
    for batch_imgs, batch_msks in aug_loader:
        batch_imgs, batch_msks = batch_imgs.to(device), batch_msks.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_imgs)
        loss = criterion(outputs, batch_msks)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(aug_loader)
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

# Save weights to separate trial path
torch.save(model.state_dict(), model_save_path)
print(f"\nTrial weights saved successfully to: {model_save_path}")

# Immediate Evaluation
model.eval()
total_iou, total_dice = 0.0, 0.0
num_samples = X_train.shape[0]
smooth = 1e-5

with torch.no_grad():
    for i in range(num_samples):
        img = X_train[i:i+1].to(device)
        true_mask = y_train[i:i+1].to(device)
        
        pred_logits = model(img)
        pred_probs = torch.sigmoid(pred_logits)
        pred_mask = (pred_probs > 0.5).float()
        
        pred_flat = pred_mask.view(-1)
        true_flat = true_mask.view(-1)
        
        intersection = (pred_flat * true_flat).sum().item()
        union = pred_flat.sum().item() + true_flat.sum().item() - intersection
        
        iou = (intersection + smooth) / (union + smooth)
        dice = (2.0 * intersection + smooth) / (pred_flat.sum().item() + true_flat.sum().item() + smooth)
        
        total_iou += iou
        total_dice += dice

trial_iou = (total_iou / num_samples) * 100
trial_dice = (total_dice / num_samples) * 100

print(f"\n--- {trial_name} Performance Metrics ---")
print(f"Mean IoU: {trial_iou:.2f}%")
print(f"Mean Dice: {trial_dice:.2f}%")

Running Experiment: trial_2_dice_bce
Epoch [1/12], Loss: 1.7738
Epoch [3/12], Loss: 1.5939
Epoch [6/12], Loss: 1.4377
Epoch [9/12], Loss: 1.3909
Epoch [12/12], Loss: 1.3713

Trial weights saved successfully to: deforestation_unet_model_trial_2_dice_bce.pth

--- trial_2_dice_bce Performance Metrics ---
Mean IoU: 16.84%
Mean Dice: 19.50%


trial -3 adamW learning rate

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import random

# Trial Identifier
trial_name = "trial_3_focal_dice"
model_save_path = f"deforestation_unet_model_{trial_name}.pth"
print(f"Running Experiment: {trial_name}")

# Re-initialize fresh model weights
model = UNet(in_channels=4, out_channels=1).to(device)

# Focal + Dice Loss for Severe Class Imbalance
class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalDiceLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets, smooth=1e-5):
        # 1. Focal Loss Component
        bce_logits = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        sigmoid_inputs = torch.sigmoid(inputs)
        p_t = targets * sigmoid_inputs + (1 - targets) * (1 - sigmoid_inputs)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * bce_logits
        focal_loss = focal_loss.mean()
        
        # 2. Dice Loss Component
        intersection = (sigmoid_inputs * targets).sum(dim=(1, 2, 3))
        total = sigmoid_inputs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice_loss = 1 - (2.0 * intersection + smooth) / (total + smooth)
        
        return focal_loss + dice_loss.mean()

# Augmentation function
def augment_batch(images, masks):
    augmented_images, augmented_masks = [], []
    for img, mask in zip(images, masks):
        if random.random() > 0.5:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[2])
        if random.random() > 0.5:
            img = torch.flip(img, dims=[1])
            mask = torch.flip(mask, dims=[1])
        augmented_images.append(img)
        augmented_masks.append(mask)
    return torch.stack(augmented_images), torch.stack(augmented_masks)

criterion = FocalDiceLoss(alpha=0.25, gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

epochs = 20
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    aug_imgs, aug_masks = augment_batch(X_train, y_train)
    aug_loader = DataLoader(TensorDataset(aug_imgs, aug_masks), batch_size=2, shuffle=True)
    
    for batch_imgs, batch_msks in aug_loader:
        batch_imgs, batch_msks = batch_imgs.to(device), batch_msks.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_imgs)
        loss = criterion(outputs, batch_msks)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(aug_loader)
    if (epoch + 1) % 4 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Focal+Dice Loss: {avg_loss:.4f}")

# Save trial weights
torch.save(model.state_dict(), model_save_path)
print(f"\nTrial weights saved to: {model_save_path}")

# Immediate Evaluation
model.eval()
total_iou, total_dice = 0.0, 0.0
num_samples = X_train.shape[0]
smooth = 1e-5

with torch.no_grad():
    for i in range(num_samples):
        img = X_train[i:i+1].to(device)
        true_mask = y_train[i:i+1].to(device)
        
        pred_logits = model(img)
        pred_probs = torch.sigmoid(pred_logits)
        pred_mask = (pred_probs > 0.5).float()
        
        pred_flat = pred_mask.view(-1)
        true_flat = true_mask.view(-1)
        
        intersection = (pred_flat * true_flat).sum().item()
        union = pred_flat.sum().item() + true_flat.sum().item() - intersection
        
        iou = (intersection + smooth) / (union + smooth)
        dice = (2.0 * intersection + smooth) / (pred_flat.sum().item() + true_flat.sum().item() + smooth)
        
        total_iou += iou
        total_dice += dice

trial_iou = (total_iou / num_samples) * 100
trial_dice = (total_dice / num_samples) * 100

print(f"\n--- {trial_name} Performance Metrics ---")
print(f"Mean IoU: {trial_iou:.2f}%")
print(f"Mean Dice: {trial_dice:.2f}%")

Running Experiment: trial_3_focal_dice
Epoch [1/20], Focal+Dice Loss: 1.0458
Epoch [4/20], Focal+Dice Loss: 1.0028
Epoch [8/20], Focal+Dice Loss: 0.9971
Epoch [12/20], Focal+Dice Loss: 0.9874
Epoch [16/20], Focal+Dice Loss: 0.9836
Epoch [20/20], Focal+Dice Loss: 0.9786

Trial weights saved to: deforestation_unet_model_trial_3_focal_dice.pth

--- trial_3_focal_dice Performance Metrics ---
Mean IoU: 9.40%
Mean Dice: 13.61%


trial-4 BCE loss with 20 epochs

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import random

# Trial Identifier
trial_name = "trial_4_optimized_bce"
model_save_path = f"deforestation_unet_model_{trial_name}.pth"
print(f"Running Experiment: {trial_name}")

# Re-initialize fresh model weights
model = UNet(in_channels=4, out_channels=1).to(device)

# Standard, highly stable BCE Loss with class weighting if needed
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-3)

# Learning rate scheduler to smoothly converge
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)

# Safe augmentation function
def augment_batch(images, masks):
    augmented_images, augmented_masks = [], []
    for img, mask in zip(images, masks):
        if random.random() > 0.5:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[2])
        if random.random() > 0.5:
            img = torch.flip(img, dims=[1])
            mask = torch.flip(mask, dims=[1])
        augmented_images.append(img)
        augmented_masks.append(mask)
    return torch.stack(augmented_images), torch.stack(augmented_masks)

epochs = 20
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    aug_imgs, aug_masks = augment_batch(X_train, y_train)
    aug_loader = DataLoader(TensorDataset(aug_imgs, aug_masks), batch_size=2, shuffle=True)
    
    for batch_imgs, batch_msks in aug_loader:
        batch_imgs, batch_msks = batch_imgs.to(device), batch_msks.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_imgs)
        loss = criterion(outputs, batch_msks)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    scheduler.step()
    avg_loss = epoch_loss / len(aug_loader)
    
    if (epoch + 1) % 4 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], BCE Loss: {avg_loss:.4f}")

# Save trial weights
torch.save(model.state_dict(), model_save_path)
print(f"\nTrial weights saved to: {model_save_path}")

# Immediate Evaluation
model.eval()
total_iou, total_dice = 0.0, 0.0
num_samples = X_train.shape[0]
smooth = 1e-5

with torch.no_grad():
    for i in range(num_samples):
        img = X_train[i:i+1].to(device)
        true_mask = y_train[i:i+1].to(device)
        
        pred_logits = model(img)
        pred_probs = torch.sigmoid(pred_logits)
        pred_mask = (pred_probs > 0.5).float()
        
        pred_flat = pred_mask.view(-1)
        true_flat = true_mask.view(-1)
        
        intersection = (pred_flat * true_flat).sum().item()
        union = pred_flat.sum().item() + true_flat.sum().item() - intersection
        
        iou = (intersection + smooth) / (union + smooth)
        dice = (2.0 * intersection + smooth) / (pred_flat.sum().item() + true_flat.sum().item() + smooth)
        
        total_iou += iou
        total_dice += dice

trial_iou = (total_iou / num_samples) * 100
trial_dice = (total_dice / num_samples) * 100

print(f"\n--- {trial_name} Performance Metrics ---")
print(f"Mean IoU: {trial_iou:.2f}%")
print(f"Mean Dice: {trial_dice:.2f}%")

Running Experiment: trial_4_optimized_bce
Epoch [1/20], BCE Loss: 0.6252
Epoch [4/20], BCE Loss: 0.3354
Epoch [8/20], BCE Loss: 0.2505
Epoch [12/20], BCE Loss: 0.2106
Epoch [16/20], BCE Loss: 0.1939
Epoch [20/20], BCE Loss: 0.1897

Trial weights saved to: deforestation_unet_model_trial_4_optimized_bce.pth

--- trial_4_optimized_bce Performance Metrics ---
Mean IoU: 56.25%
Mean Dice: 56.25%


Trial-5

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import random
import numpy as np

# Trial Identifier
trial_name = "trial_5_high_accuracy"
model_save_path = f"deforestation_unet_model_{trial_name}.pth"
print(f"Running Experiment: {trial_name}")

# 1. Explicit Train/Test Split (80% Train, 20% Test)
torch.manual_seed(42)
num_total = X_train.shape[0]
indices = torch.randperm(num_total)
split_idx = int(0.8 * num_total)

train_indices = indices[:split_idx]
test_indices = indices[split_idx:]

X_tr, y_tr = X_train[train_indices], y_train[train_indices]
X_te, y_te = X_train[test_indices], y_train[test_indices]

print(f"Dataset Split -> Training samples: {X_tr.shape[0]}, Test samples: {X_te.shape[0]}")

# 2. Re-initialize fresh model weights
model = UNet(in_channels=4, out_channels=1).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0008, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

# Augmentation function
def augment_batch(images, masks):
    augmented_images, augmented_masks = [], []
    for img, mask in zip(images, masks):
        if random.random() > 0.5:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[2])
        if random.random() > 0.5:
            img = torch.flip(img, dims=[1])
            mask = torch.flip(mask, dims=[1])
        augmented_images.append(img)
        augmented_masks.append(mask)
    return torch.stack(augmented_images), torch.stack(augmented_masks)

# 3. Training Loop (30 Epochs)
epochs = 30
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    aug_imgs, aug_masks = augment_batch(X_tr, y_tr)
    aug_loader = DataLoader(TensorDataset(aug_imgs, aug_masks), batch_size=2, shuffle=True)
    
    for batch_imgs, batch_msks in aug_loader:
        batch_imgs, batch_msks = batch_imgs.to(device), batch_msks.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_imgs)
        loss = criterion(outputs, batch_msks)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    scheduler.step()
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        avg_loss = epoch_loss / len(aug_loader)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

# Save Trial 5 weights
torch.save(model.state_dict(), model_save_path)
print(f"\nTrial weights saved to: {model_save_path}")

# 4. Evaluation Function for Metrics
def evaluate_model(images, masks, dataset_name):
    model.eval()
    total_iou, total_dice = 0.0, 0.0
    n = images.shape[0]
    smooth = 1e-5
    
    with torch.no_grad():
        for i in range(n):
            img = images[i:i+1].to(device)
            true_mask = masks[i:i+1].to(device)
            
            pred_logits = model(img)
            pred_probs = torch.sigmoid(pred_logits)
            pred_mask = (pred_probs > 0.5).float()
            
            pred_flat = pred_mask.view(-1)
            true_flat = true_mask.view(-1)
            
            intersection = (pred_flat * true_flat).sum().item()
            union = pred_flat.sum().item() + true_flat.sum().item() - intersection
            
            iou = (intersection + smooth) / (union + smooth)
            dice = (2.0 * intersection + smooth) / (pred_flat.sum().item() + true_flat.sum().item() + smooth)
            
            total_iou += iou
            total_dice += dice
            
    mean_iou = (total_iou / n) * 100
    mean_dice = (total_dice / n) * 100
    print(f"\n--- {dataset_name} Performance ---")
    print(f"Mean IoU: {mean_iou:.2f}%")
    print(f"Mean Dice (F1): {mean_dice:.2f}%")
    return mean_iou, mean_dice

# Run Separate Evaluations
train_iou, train_dice = evaluate_model(X_tr, y_tr, "Training Set")
test_iou, test_dice = evaluate_model(X_te, y_te, "Held-Out Test Set")

Running Experiment: trial_5_high_accuracy
Dataset Split -> Training samples: 12, Test samples: 4
Epoch [1/30], Loss: 0.5409
Epoch [5/30], Loss: 0.2392
Epoch [10/30], Loss: 0.1549
Epoch [15/30], Loss: 0.1175
Epoch [20/30], Loss: 0.1011
Epoch [25/30], Loss: 0.0943
Epoch [30/30], Loss: 0.0931

Trial weights saved to: deforestation_unet_model_trial_5_high_accuracy.pth

--- Training Set Performance ---
Mean IoU: 58.33%
Mean Dice (F1): 58.33%

--- Held-Out Test Set Performance ---
Mean IoU: 50.00%
Mean Dice (F1): 50.00%


Trial- 6

In [21]:
import ee
import geemap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import random
import numpy as np

# Trial Identifier
trial_name = "trial_6_grid_data"
model_save_path = f"deforestation_unet_model_{trial_name}.pth"
print(f"Running Experiment: {trial_name}")

ee.Initialize(project='peppy-gist-511817-e8')

# Define a 3x3 grid of tiny sub-regions to stay safely under GEE's 50MB limit
lat_mins = [-10.6, -10.5, -10.4]
lat_maxs = [-10.5, -10.4, -10.3]
lon_mins = [-59.8, -59.7, -59.6]
lon_maxs = [-59.7, -59.6, -59.5]

sub_aois = []
for lat_min, lat_max in zip(lat_mins, lat_maxs):
    for lon_min, lon_max in zip(lon_mins, lon_maxs):
        sub_aois.append(ee.Geometry.Polygon([[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max]
        ]]))

all_images, all_masks = [], []

for idx, sub_aoi in enumerate(sub_aois):
    print(f"Extracting tiny sub-region {idx+1}/9 from GEE...")
    
    collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(sub_aoi) \
        .filterDate('2024-06-01', '2024-10-31') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    image_2024 = collection.median().clip(sub_aoi)

    image_2022 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(sub_aoi) \
        .filterDate('2022-06-01', '2022-10-31') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
        .median().clip(sub_aoi)

    ndvi_2022 = image_2022.normalizedDifference(['B8', 'B4'])
    ndvi_2024 = image_2024.normalizedDifference(['B8', 'B4'])
    ndvi_diff = ndvi_2022.subtract(ndvi_2024)
    deforestation_mask = ndvi_diff.gt(0.25)

    cleaned_deforestation = (deforestation_mask
        .selfMask()
        .connectedPixelCount(maxSize=50, eightConnected=True)
        .gte(10))

    s2_rgbn = image_2024.select(['B4', 'B3', 'B2', 'B8'])
    mask_image = cleaned_deforestation.selfMask().unmask(0).rename('label')
    training_image = s2_rgbn.addBands(mask_image)

    try:
        region_array = geemap.ee_to_numpy(training_image, region=sub_aoi, scale=10)
        patch_size = 256
        h, w, c = region_array.shape
        if h >= patch_size and w >= patch_size:
            for y in range(0, h - patch_size + 1, patch_size):
                for x in range(0, w - patch_size + 1, patch_size):
                    patch = np.nan_to_num(region_array[y:y+patch_size, x:x+patch_size, :])
                    all_images.append(patch[:, :, :4].transpose(2, 0, 1))
                    all_masks.append(patch[:, :, 4:5].transpose(2, 0, 1))
    except Exception as e:
        print(f"Skipping tile {idx+1} due to empty/error payload: {e}")

X_full = torch.tensor(np.array(all_images), dtype=torch.float32)
y_full = torch.tensor(np.array(all_masks), dtype=torch.float32)
print(f"Total extracted patches across grid: {X_full.shape[0]}")

# 1. Explicit Train/Test Split (80% Train, 20% Test)
torch.manual_seed(42)
num_total = X_full.shape[0]
indices = torch.randperm(num_total)
split_idx = int(0.8 * num_total)

X_tr, y_tr = X_full[indices[:split_idx]], y_full[indices[:split_idx]]
X_te, y_te = X_full[indices[split_idx:]], y_full[indices[split_idx:]]
print(f"Training samples: {X_tr.shape[0]}, Test samples: {X_te.shape[0]}")

# 2. Fresh Model & Training Setup
model = UNet(in_channels=4, out_channels=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0008, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=35, eta_min=1e-6)

def augment_batch(images, masks):
    augmented_images, augmented_masks = [], []
    for img, mask in zip(images, masks):
        if random.random() > 0.5:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[2])
        if random.random() > 0.5:
            img = torch.flip(img, dims=[1])
            mask = torch.flip(mask, dims=[1])
        augmented_images.append(img)
        augmented_masks.append(mask)
    return torch.stack(augmented_images), torch.stack(augmented_masks)

epochs = 35
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    aug_imgs, aug_masks = augment_batch(X_tr, y_tr)
    aug_loader = DataLoader(TensorDataset(aug_imgs, aug_masks), batch_size=4, shuffle=True)
    
    for batch_imgs, batch_msks in aug_loader:
        batch_imgs, batch_msks = batch_imgs.to(device), batch_msks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch_imgs), batch_msks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    scheduler.step()
    if (epoch + 1) % 7 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss / len(aug_loader):.4f}")

torch.save(model.state_dict(), model_save_path)
print(f"\nTrial weights saved to: {model_save_path}")

# 3. Evaluation
def evaluate_model(images, masks, dataset_name):
    model.eval()
    total_iou, total_dice = 0.0, 0.0
    n = images.shape[0]
    smooth = 1e-5
    
    with torch.no_grad():
        for i in range(n):
            img = images[i:i+1].to(device)
            true_mask = masks[i:i+1].to(device)
            
            pred_logits = model(img)
            pred_mask = (torch.sigmoid(pred_logits) > 0.5).float()
            
            pred_flat = pred_mask.view(-1)
            true_flat = true_mask.view(-1)
            
            intersection = (pred_flat * true_flat).sum().item()
            union = pred_flat.sum().item() + true_flat.sum().item() - intersection
            
            total_iou += (intersection + smooth) / (union + smooth)
            total_dice += (2.0 * intersection + smooth) / (pred_flat.sum().item() + true_flat.sum().item() + smooth)
            
    mean_iou = (total_iou / n) * 100
    mean_dice = (total_dice / n) * 100
    print(f"\n--- {dataset_name} Performance ---")
    print(f"Mean IoU: {mean_iou:.2f}%")
    print(f"Mean Dice (F1): {mean_dice:.2f}%")
    return mean_iou, mean_dice

evaluate_model(X_tr, y_tr, "Training Set")
evaluate_model(X_te, y_te, "Held-Out Test Set")

Running Experiment: trial_6_grid_data
Extracting tiny sub-region 1/9 from GEE...
Extracting tiny sub-region 2/9 from GEE...
Extracting tiny sub-region 3/9 from GEE...
Extracting tiny sub-region 4/9 from GEE...
Extracting tiny sub-region 5/9 from GEE...
Extracting tiny sub-region 6/9 from GEE...
Extracting tiny sub-region 7/9 from GEE...
Extracting tiny sub-region 8/9 from GEE...
Extracting tiny sub-region 9/9 from GEE...
Total extracted patches across grid: 144
Training samples: 115, Test samples: 29
Epoch [1/35], Loss: 0.4832
Epoch [7/35], Loss: 0.0629
Epoch [14/35], Loss: 0.0294
Epoch [21/35], Loss: 0.0241
Epoch [28/35], Loss: 0.0211
Epoch [35/35], Loss: 0.0201

Trial weights saved to: deforestation_unet_model_trial_6_grid_data.pth

--- Training Set Performance ---
Mean IoU: 57.39%
Mean Dice (F1): 57.39%

--- Held-Out Test Set Performance ---
Mean IoU: 65.52%
Mean Dice (F1): 65.52%


(65.51724723105681, 65.51724723105681)

trial -7

In [22]:
# Trial Identifier
trial_name = "trial_7_final_push"
model_save_path = f"deforestation_unet_model_{trial_name}.pth"
print(f"Running Experiment: {trial_name}")

# 1. Reuse existing X_full and y_full with 80/20 split
torch.manual_seed(42)
num_total = X_full.shape[0]
indices = torch.randperm(num_total)
split_idx = int(0.8 * num_total)

X_tr, y_tr = X_full[indices[:split_idx]], y_full[indices[:split_idx]]
X_te, y_te = X_full[indices[split_idx:]], y_full[indices[split_idx:]]
print(f"Training samples: {X_tr.shape[0]}, Test samples: {X_te.shape[0]}")

# 2. Fresh Model & Optimized Setup
model = UNet(in_channels=4, out_channels=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

def augment_batch(images, masks):
    augmented_images, augmented_masks = [], []
    for img, mask in zip(images, masks):
        if random.random() > 0.5:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[2])
        if random.random() > 0.5:
            img = torch.flip(img, dims=[1])
            mask = torch.flip(mask, dims=[1])
        augmented_images.append(img)
        augmented_masks.append(mask)
    return torch.stack(augmented_images), torch.stack(augmented_masks)

# 3. Extended Training Loop (50 Epochs)
epochs = 50
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    aug_imgs, aug_masks = augment_batch(X_tr, y_tr)
    aug_loader = DataLoader(TensorDataset(aug_imgs, aug_masks), batch_size=4, shuffle=True)
    
    for batch_imgs, batch_msks in aug_loader:
        batch_imgs, batch_msks = batch_imgs.to(device), batch_msks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch_imgs), batch_msks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    scheduler.step()
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss / len(aug_loader):.4f}")

torch.save(model.state_dict(), model_save_path)
print(f"\nTrial weights saved to: {model_save_path}")

# 4. Rigorous Evaluation Function
def evaluate_model(images, masks, dataset_name):
    model.eval()
    total_iou, total_dice = 0.0, 0.0
    n = images.shape[0]
    smooth = 1e-5
    
    with torch.no_grad():
        for i in range(n):
            img = images[i:i+1].to(device)
            true_mask = masks[i:i+1].to(device)
            
            pred_logits = model(img)
            pred_mask = (torch.sigmoid(pred_logits) > 0.5).float()
            
            pred_flat = pred_mask.view(-1)
            true_flat = true_mask.view(-1)
            
            intersection = (pred_flat * true_flat).sum().item()
            union = pred_flat.sum().item() + true_flat.sum().item() - intersection
            
            total_iou += (intersection + smooth) / (union + smooth)
            total_dice += (2.0 * intersection + smooth) / (pred_flat.sum().item() + true_flat.sum().item() + smooth)
            
    mean_iou = (total_iou / n) * 100
    mean_dice = (total_dice / n) * 100
    print(f"\n--- {dataset_name} Performance ---")
    print(f"Mean IoU: {mean_iou:.2f}%")
    print(f"Mean Dice (F1): {mean_dice:.2f}%")
    return mean_iou, mean_dice

evaluate_model(X_tr, y_tr, "Training Set")
evaluate_model(X_te, y_te, "Held-Out Test Set")

Running Experiment: trial_7_final_push
Training samples: 115, Test samples: 29
Epoch [1/50], Loss: 0.4624
Epoch [10/50], Loss: 0.0317
Epoch [20/50], Loss: 0.0214
Epoch [30/50], Loss: 0.0182
Epoch [40/50], Loss: 0.0166
Epoch [50/50], Loss: 0.0160

Trial weights saved to: deforestation_unet_model_trial_7_final_push.pth

--- Training Set Performance ---
Mean IoU: 57.39%
Mean Dice (F1): 57.39%

--- Held-Out Test Set Performance ---
Mean IoU: 65.52%
Mean Dice (F1): 65.52%


(65.51724723105681, 65.51724723105681)

Trial -8

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import random

# Trial Identifier
trial_name = "trial_8_iou_optimized"
model_save_path = f"deforestation_unet_model_{trial_name}.pth"
print(f"Running Experiment: {trial_name}")

# 1. Combined BCE + Soft Dice Loss for direct overlap optimization
class ComboLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super(ComboLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, inputs, targets, smooth=1e-5):
        bce_loss = self.bce(inputs, targets)
        
        sigmoid_inputs = torch.sigmoid(inputs)
        intersection = (sigmoid_inputs * targets).sum(dim=(1, 2, 3))
        total = sigmoid_inputs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice_loss = 1 - (2.0 * intersection + smooth) / (total + smooth)
        
        return (self.bce_weight * bce_loss) + (self.dice_weight * dice_loss.mean())

# 2. Fresh Model & Optimized Setup
model = UNet(in_channels=4, out_channels=1).to(device)
criterion = ComboLoss(bce_weight=0.4, dice_weight=0.6)
optimizer = optim.AdamW(model.parameters(), lr=0.0008, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=45, eta_min=1e-6)

def augment_batch(images, masks):
    augmented_images, augmented_masks = [], []
    for img, mask in zip(images, masks):
        if random.random() > 0.5:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[2])
        if random.random() > 0.5:
            img = torch.flip(img, dims=[1])
            mask = torch.flip(mask, dims=[1])
        augmented_images.append(img)
        augmented_masks.append(mask)
    return torch.stack(augmented_images), torch.stack(augmented_masks)

# 3. Training Loop (45 Epochs)
epochs = 45
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    aug_imgs, aug_masks = augment_batch(X_tr, y_tr)
    aug_loader = DataLoader(TensorDataset(aug_imgs, aug_masks), batch_size=4, shuffle=True)
    
    for batch_imgs, batch_msks in aug_loader:
        batch_imgs, batch_msks = batch_imgs.to(device), batch_msks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch_imgs), batch_msks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    scheduler.step()
    if (epoch + 1) % 9 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Combo Loss: {epoch_loss / len(aug_loader):.4f}")

torch.save(model.state_dict(), model_save_path)
print(f"\nTrial weights saved to: {model_save_path}")

# 4. Evaluation with Optimal Threshold Search
def evaluate_with_best_threshold(images, masks, dataset_name):
    model.eval()
    smooth = 1e-5
    n = images.shape[0]
    
    # Collect all predictions and targets first
    all_probs = []
    with torch.no_grad():
        for i in range(n):
            img = images[i:i+1].to(device)
            pred_logits = model(img)
            pred_probs = torch.sigmoid(pred_logits)
            all_probs.append(pred_probs.cpu())
    
    all_probs = torch.cat(all_probs, dim=0)
    cpu_masks = masks.cpu()
    
    # Test different thresholds to find the highest IoU
    best_thresh = 0.5
    best_iou = 0.0
    
    for thresh in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
        total_iou = 0.0
        for i in range(n):
            pred_mask = (all_probs[i] > thresh).float()
            true_mask = cpu_masks[i]
            
            pred_flat = pred_mask.view(-1)
            true_flat = true_mask.view(-1)
            
            intersection = (pred_flat * true_flat).sum().item()
            union = pred_flat.sum().item() + true_flat.sum().item() - intersection
            total_iou += (intersection + smooth) / (union + smooth)
            
        mean_iou = (total_iou / n) * 100
        if mean_iou > best_iou:
            best_iou = mean_iou
            best_thresh = thresh
            
    # Calculate final Dice using best threshold
    total_dice = 0.0
    for i in range(n):
        pred_mask = (all_probs[i] > best_thresh).float()
        true_mask = cpu_masks[i]
        pred_flat = pred_mask.view(-1)
        true_flat = true_mask.view(-1)
        intersection = (pred_flat * true_flat).sum().item()
        total_dice += (2.0 * intersection + smooth) / (pred_flat.sum().item() + true_flat.sum().item() + smooth)
        
    mean_dice = (total_dice / n) * 100
    
    print(f"\n--- {dataset_name} Performance (Best Thresh: {best_thresh}) ---")
    print(f"Mean IoU: {best_iou:.2f}%")
    print(f"Mean Dice (F1): {mean_dice:.2f}%")
    return best_iou, mean_dice

evaluate_with_best_threshold(X_tr, y_tr, "Training Set")
evaluate_with_best_threshold(X_te, y_te, "Held-Out Test Set")

Running Experiment: trial_8_iou_optimized
Epoch [1/45], Combo Loss: 0.7827
Epoch [9/45], Combo Loss: 0.6114
Epoch [18/45], Combo Loss: 0.5864
Epoch [27/45], Combo Loss: 0.5796
Epoch [36/45], Combo Loss: 0.5724
Epoch [45/45], Combo Loss: 0.5729

Trial weights saved to: deforestation_unet_model_trial_8_iou_optimized.pth

--- Training Set Performance (Best Thresh: 0.6) ---
Mean IoU: 40.56%
Mean Dice (F1): 43.46%

--- Held-Out Test Set Performance (Best Thresh: 0.6) ---
Mean IoU: 35.71%
Mean Dice (F1): 38.06%


(35.709021849769016, 38.063998466183136)